# Make it fail: the controls, with reasons

A check that cannot fail shows nothing. This notebook removes one hypothesis of Theorem 2 at a time and shows the two sides of the recovery result disagreeing, in the direction the theory predicts, with the reason printed next to each verdict. As in the walkthrough, every function comes from `issrdf` (or, in the last section, from the owlrl check script); the notebook computes nothing itself.

1. **The Corollary 3 slip**: closing the graph with rule instances over `N` only. An earlier draft of the paper did this; the semantics says *yes* and the rules say *no*.
2. **A non-uniform regime**: a rule that fires only for blank-node subjects, violating Definition 4(2). The rules say *yes* and the semantics says *no*.
3. **An undersized vocabulary with a real reasoner**: owlrl's OWL 2 RL rule set with `V` taken as the hand-listed OWL vocabulary rather than the regime's real vocabulary. Admissibility (Definition 12) fails, and so does agreement. This section is skipped if owlrl is not installed.

In [1]:
import sys; sys.path.insert(0, '..')
from issrdf import (Universe, BOT, make_closure, simply_entails, r_entails, r_inconsistent, inst,
                    mappings, instances, content_pos, content_neg, adj, make_good, iss_entails, show)
from issrdf.show import triple, graph, pairs, regime, mapping, report, why, verdict_line
verdicts = []

## 1. The Corollary 3 slip: rule instances over `N` only

The closure cl_R(X) of Definition 4 is closure under *every* instance of the regime's rules, including instances at blank nodes: the regime is closed under all maps ρ on I ∪ B ∪ L fixing L ∪ V (uniformity), so a rule instance with `_:x` in place of `X` is as much a rule as one with `C`. An earlier draft of Corollary 3 took the instances over `N` only. Here is what that does.

The regime is rdfs9 (cax-sco in the OWL 2 RL naming), `{(X, type, Y), (Y, subClassOf, Z)} → (X, type, Z)`. `G` = {(_:x, type, C), (C, subClassOf, D)}, `H` = {(_:y, type, D)}. Blank node labels are local to a graph (Remark 2), so `H`'s blank node is written `_:y` rather than reusing `_:x`; the paper's statement of the counterexample writes both as `_:x`, and nothing depends on the choice, since Definition 3's instance mapping and Definition 11's mappings each treat `H`'s blank node on its own.

Under the correct closure the instance of rdfs9 at `X = _:x` fires, cl_R(G) contains (_:x, type, D), and μ(_:y) = _:x witnesses simple entailment. Under instances over `N` only, that instance is not available, cl(G) = G, and no instance mapping into terms(G) makes (μ(_:y), type, D) a member.

The semantics' side is unaffected, and that is the point. The frame 𝕀_R is built from the base 𝓑_R, whose sequents are ground triples over `N` (Definition 13). For a ground Γ over `N`, the instances that can fire are over names(Γ) ∪ V ⊆ N anyway (Lemma 6), so restricting instances to `N` changes nothing on the semantics' side. The restriction removes exactly the instances at blank nodes, which only the rules' side needs, so the two come apart: **semantics yes, rules no**.

In [2]:
U1 = Universe(V=('type', 'subClassOf'), INDIV=('C', 'D'), SPARE=('s',), BN_G=('_x',), BN_H=('_y',))
R1 = (((('X', 'type', 'Y'), ('Y', 'subClassOf', 'Z')), ('X', 'type', 'Z')),)     # rdfs9 / cax-sco
G1 = frozenset({('_x', 'type', 'C'), ('C', 'subClassOf', 'D')})
H1 = frozenset({('_y', 'type', 'D')})
assert U1.admissible(G1, H1)
print(U1); print(regime(R1)); print("G =", graph(G1)); print("H =", graph(H1))

cl_all = make_closure(R1, U1.V)                        # Definition 4
cl_N = make_closure(R1, U1.V, instances_over=U1.N)      # the slip: instances over N only
good = make_good(cl_all)                                # the frame, Definitions 13-14

print("\n-- (a) all instances (Definition 4) --")
print("cl_R(G) \\ G =", graph(cl_all(G1) - G1))
print("   the instance of rdfs9 that fired:", show.rule((tuple(inst(t, {'X': '_x', 'Y': 'C', 'Z': 'D'}) for t in R1[0][0]), inst(R1[0][1], {'X': '_x', 'Y': 'C', 'Z': 'D'}))))
rules_a = r_entails(cl_all, G1, H1)
print("some μ with μ(H) ⊆ cl_R(G):", rules_a, " (μ(_:y) = _:x)")
sem_a = iss_entails(good, G1, H1, U1)
print("semantics: every pair of [[G]]+ ⊔ [[H]]- in I_R:", sem_a)
print(verdict_line("1a: all instances", sem_a, rules_a)); verdicts.append(("1a: Cor. 3 example, all instances", sem_a, rules_a))

print("\n-- (b) instances over N only --")
print("instances are taken over N =", U1.N, "; _:x is not in N, so the instance above is not among them")
print("cl_N(G) \\ G =", graph(cl_N(G1) - G1))
rules_b = r_entails(cl_N, G1, H1)
print("some μ with μ(H) ⊆ cl_N(G):", rules_b)
print("\nsemantics' side, unchanged: the closure of each ground instance ν(G) over N is the same under both closures:")
for nu, g in zip(mappings(G1, U1), instances(G1, U1)):
    print(f"   ν = {mapping(nu):16s} cl_R(ν(G)) \\ ν(G) = {graph(cl_all(g) - g):24s} cl_N(ν(G)) \\ ν(G) = {graph(cl_N(g) - g)}")
sem_b = iss_entails(make_good(cl_N), G1, H1, U1)
assert sem_b == sem_a
print("semantics (frame built from cl_N, same result):", sem_b)
print(verdict_line("1b: instances over N only", sem_b, rules_b)); verdicts.append(("1b: Cor. 3 example, instances over N only", sem_b, rules_b))

Universe(V=('type', 'subClassOf'), INDIV=('C', 'D'), SPARE=('s',), BN_G=('_x',), BN_H=('_y',))  # N = ('C', 'D', 'type', 'subClassOf', 's')
{(X, type, Y), (Y, subClassOf, Z)} → (X, type, Z)
G = {(C, subClassOf, D), (_:x, type, C)}
H = {(_:y, type, D)}

-- (a) all instances (Definition 4) --
cl_R(G) \ G = {(_:x, type, D)}
   the instance of rdfs9 that fired: {(_:x, type, C), (C, subClassOf, D)} → (_:x, type, D)
some μ with μ(H) ⊆ cl_R(G): True  (μ(_:y) = _:x)
semantics: every pair of [[G]]+ ⊔ [[H]]- in I_R: True
1a: all instances                        semantics True   rules True   agree

-- (b) instances over N only --
instances are taken over N = ('C', 'D', 'type', 'subClassOf', 's') ; _:x is not in N, so the instance above is not among them
cl_N(G) \ G = ∅
some μ with μ(H) ⊆ cl_N(G): False

semantics' side, unchanged: the closure of each ground instance ν(G) over N is the same under both closures:
   ν = {_:x ↦ C}        cl_R(ν(G)) \ ν(G) = {(C, type, D)}           cl_N(ν(G)) \ ν(G) 

## 2. A non-uniform regime

Definition 4(2) requires the regime to be closed under every map ρ on I ∪ B ∪ L that fixes L ∪ V pointwise: if ⟨A, c⟩ is a rule then so is ⟨ρ(A), ρ(c)⟩. Rule schemas without side conditions satisfy this automatically (Remark 5). Here is a regime that does not: the single rule `{(X, type, Bird)} → (X, type, Flier)`, but firing **only when `X` is a blank node**. Its instance at `_:x` is a rule; the image of that instance under ρ(_:x) = tweety is not.

`G` = {(_:x, type, Bird)}, `H` = {(_:y, type, Flier)}.

**Rules' side.** `X = _:x` is a blank node, so the rule fires: cl(G) contains (_:x, type, Flier), and μ(_:y) = _:x witnesses `H`. Verdict *yes*.

**Semantics' side.** The base 𝓑_R (Definition 13) is a relation on ground triples over `N`. Every instance ν(G) is ground, so the rule never fires on any Γ the frame is built from: cl(ν(G)) = ν(G) for all five ν, and no instance of `H` is in it. Every generating pair fails. Verdict *no*. The rule is invisible to the frame because it only ever applies to things the frame does not contain.

What breaks in the proof is Lemma 6, ρ(cl_R(X)) ⊆ cl_R(ρ(X)), which is where uniformity is used: with ρ(_:x) = tweety, ρ(cl(G)) contains (tweety, type, Flier) but cl(ρ(G)) does not. The direction is **rules yes, semantics no**. The same rule as a uniform schema, firing for every `X`, restores agreement.

In [3]:
U2 = Universe(V=('type', 'Bird', 'Flier'), INDIV=('tweety',), SPARE=('s',), BN_G=('_x',), BN_H=('_y',))
R2 = (((('X', 'type', 'Bird'),), ('X', 'type', 'Flier')),)
G2 = frozenset({('_x', 'type', 'Bird')})
H2 = frozenset({('_y', 'type', 'Flier')})
assert U2.admissible(G2, H2)
print(U2); print(regime(R2), "  -- firing only when X is a blank node"); print("G =", graph(G2)); print("H =", graph(H2))

cl_nu = make_closure(R2, U2.V, bnode_only=True)     # the non-uniform regime
good_nu = make_good(cl_nu)                           # the frame it induces (Definitions 13-14)

print("\n-- rules' side --")
print("cl(G) \\ G =", graph(cl_nu(G2) - G2))
rules_2 = r_entails(cl_nu, G2, H2)
print("some μ with μ(H) ⊆ cl(G):", rules_2, " (μ(_:y) = _:x)")

print("\n-- semantics' side --")
print("the ground instances ν(G) over N, and their closures under the non-uniform regime:")
for nu, g in zip(mappings(G2, U2), instances(G2, U2)):
    print(f"   ν = {mapping(nu):16s} ν(G) = {graph(g):24s} cl(ν(G)) \\ ν(G) = {graph(cl_nu(g) - g)}")
F = adj(content_pos(G2, U2), content_neg(H2, U2))
print(f"[[G]]+ ⊔ [[H]]-: {len(F)} pairs; the 5 singleton pairs <ν(G), Δ>, each tested against I_R:")
report(cl_nu, [p for p in F if len(p[0]) == 1])
sem_2 = iss_entails(good_nu, G2, H2, U2)
print("all pairs in I_R:", sem_2)
print(verdict_line("2: non-uniform regime", sem_2, rules_2)); verdicts.append(("2: non-uniform regime", sem_2, rules_2))

print("\n-- where the proof breaks: Lemma 6 with ρ(_:x) = tweety --")
rho = {'_x': 'tweety'}
lhs = frozenset(inst(t, rho) for t in cl_nu(G2)); rhs = cl_nu(frozenset(inst(t, rho) for t in G2))
print("ρ(cl(G)) =", graph(lhs)); print("cl(ρ(G)) =", graph(rhs)); print("ρ(cl(G)) ⊆ cl(ρ(G)):", lhs <= rhs)

print("\n-- the same rule as a uniform schema --")
cl_u = make_closure(R2, U2.V)
sem_u = iss_entails(make_good(cl_u), G2, H2, U2); rules_u = r_entails(cl_u, G2, H2)
print(verdict_line("2': same rule, uniform", sem_u, rules_u)); verdicts.append(("2': same rule, uniform", sem_u, rules_u))

Universe(V=('type', 'Bird', 'Flier'), INDIV=('tweety',), SPARE=('s',), BN_G=('_x',), BN_H=('_y',))  # N = ('tweety', 'type', 'Bird', 'Flier', 's')
{(X, type, Bird)} → (X, type, Flier)   -- firing only when X is a blank node
G = {(_:x, type, Bird)}
H = {(_:y, type, Flier)}

-- rules' side --
cl(G) \ G = {(_:x, type, Flier)}
some μ with μ(H) ⊆ cl(G): True  (μ(_:y) = _:x)

-- semantics' side --
the ground instances ν(G) over N, and their closures under the non-uniform regime:
   ν = {_:x ↦ tweety}   ν(G) = {(tweety, type, Bird)}   cl(ν(G)) \ ν(G) = ∅
   ν = {_:x ↦ type}     ν(G) = {(type, type, Bird)}     cl(ν(G)) \ ν(G) = ∅
   ν = {_:x ↦ Bird}     ν(G) = {(Bird, type, Bird)}     cl(ν(G)) \ ν(G) = ∅
   ν = {_:x ↦ Flier}    ν(G) = {(Flier, type, Bird)}    cl(ν(G)) \ ν(G) = ∅
   ν = {_:x ↦ s}        ν(G) = {(s, type, Bird)}        cl(ν(G)) \ ν(G) = ∅
[[G]]+ ⊔ [[H]]-: 31 pairs; the 5 singleton pairs <ν(G), Δ>, each tested against I_R:
  ✗ ⟨{(Bird, type, Bird)}, {(Bird, type, Flier), (Flier, 

## 3. An undersized vocabulary, with owlrl

The end-to-end check (`checks/check_owlrl.py`) uses owlrl, a deployed OWL 2 RL reasoner, as the closure on both sides. The semantics' side needs `N`, and `N` needs the regime's vocabulary `V` (Definition 13: "the requirement V ⊆ N ensures that no rule instance is excluded by the restriction to N"; Definition 12 requires it for admissibility). What *is* the vocabulary of owlrl's rule set? Not just the OWL and RDFS IRIs one would list by hand. With axiomatic triples switched off, owlrl still adds 106 triples over 54 IRIs to the empty graph: XSD datatypes typed as `rdfs:Datatype`, the OWL annotation properties typed as `owl:AnnotationProperty`, and their `owl:sameAs` reflexivity. Each is a rule with empty premises, ⟨∅, t⟩, and by range restriction (Definition 4(1)) every name in t belongs to `V`. Forty-one of those 54 IRIs are not in the hand list.

A logged random search (`results/undersized_V_search.txt`: 120 OWL cases with the hand-listed `V`, one mismatch, minimized by deleting triples) gives the smallest case: `G` = ∅ and `H` = {(owl:deprecated, rdf:type, _:y)}, "owl:deprecated has some type". The rules derive (owl:deprecated, type, owl:AnnotationProperty) from nothing, so the SPARQL ASK succeeds. On the semantics' side μ(_:y) ranges over `N` = names(H) ∪ V ∪ {s}, and with the hand-listed `V`, `owl:AnnotationProperty` is not in `N`: no instance of `H` is in the closure, and the verdict is *no*. With `V` completed by the IRIs of owlrl's closure of the empty graph, `N` is admissible and the two sides agree. This is the only kind of mismatch the owlrl runs ever produced (README, "lessons").

In [4]:
try:
    sys.path.insert(0, '../checks')
    import check_owlrl as W
    from rdflib import BNode, URIRef
    HAVE_OWLRL = True
except ImportError as e:
    HAVE_OWLRL = False
    print("skipped: owlrl/rdflib not installed (pip install -r requirements.txt) --", e)

if HAVE_OWLRL:
    sem = W.OWLRL_Semantics
    V_hand = set(W.V_OWL)
    g0, _ = W.close(frozenset(), sem)                    # owlrl's closure of the empty graph
    V_full = V_hand | {x for tr in g0 for x in tr if isinstance(x, URIRef)}
    short = lambda t: '_:' + str(t) if isinstance(t, BNode) else str(t).split('#')[-1].split('/')[-1]
    fmt = lambda G: '{' + ', '.join('(' + ', '.join(short(u) for u in tr) + ')' for tr in sorted(G, key=str)) + '}' if G else '∅'
    print(f"hand-listed V: {len(V_hand)} IRIs;  owlrl's closure of ∅: {len(g0)} triples over "
          f"{len({x for tr in g0 for x in tr})} IRIs, of which {len(V_full - V_hand)} are outside the hand list;  V_full: {len(V_full)} IRIs")

    G3 = frozenset(); H3 = frozenset({(W.OWL.deprecated, W.RDF.type, W.BY)})
    print("G =", fmt(G3), "  H =", fmt(H3))
    wit = [t for t in g0 if t[0] == W.OWL.deprecated and t[1] == W.RDF.type]
    print("in the closure of G (= closure of ∅):", fmt(wit), " -- an axiomatic rule <∅, t> of owlrl's regime")

    def N_for(G, H, V):
        names = {t for tr in G | H for t in tr if not isinstance(t, BNode)}
        return sorted(names | set(V) | {W.SPARE}, key=str)
    for label, desc, V in [("hand-listed V", "the hand-listed OWL vocabulary", V_hand),
                           ("V_full", "hand list + the IRIs of owlrl's closure of ∅", V_full)]:
        N = N_for(G3, H3, V)
        print(f"\n-- {label} ({desc}): |N| = {len(N)}; owl:AnnotationProperty ∈ N: {W.OWL.AnnotationProperty in N} --")
        rules_3 = W.def4_side(G3, H3, sem)                  # materialize, then ASK (μ unrestricted)
        sem_3 = W.iss_side(G3, H3, sem, N)                  # every ν(G) over N: some μ into N with μ(H) ⊆ cl(ν(G))
        print("rules:     ASK { owl:deprecated rdf:type ?y } over the closure:", rules_3)
        print("semantics: some μ : {_:y} → N with μ(H) ⊆ cl(G):", sem_3)
        print(verdict_line(f"3: owlrl, {label}", sem_3, rules_3))
        verdicts.append((f"3: owlrl, {label}", sem_3, rules_3))

hand-listed V: 62 IRIs;  owlrl's closure of ∅: 106 triples over 54 IRIs, of which 41 are outside the hand list;  V_full: 103 IRIs
G = ∅   H = {(deprecated, type, _:y)}
in the closure of G (= closure of ∅): {(deprecated, type, AnnotationProperty)}  -- an axiomatic rule <∅, t> of owlrl's regime

-- hand-listed V (the hand-listed OWL vocabulary): |N| = 64; owl:AnnotationProperty ∈ N: False --
rules:     ASK { owl:deprecated rdf:type ?y } over the closure: True
semantics: some μ : {_:y} → N with μ(H) ⊆ cl(G): False
3: owlrl, hand-listed V                  semantics False  rules True   DISAGREE

-- V_full (hand list + the IRIs of owlrl's closure of ∅): |N| = 104; owl:AnnotationProperty ∈ N: True --
rules:     ASK { owl:deprecated rdf:type ?y } over the closure: True
semantics: some μ : {_:y} → N with μ(H) ⊆ cl(G): True
3: owlrl, V_full                         semantics True   rules True   agree


## Summary

Each removed hypothesis produces a disagreement, and each in the predicted direction; restoring it (1a, 2', the second row of 3) restores agreement. The last cell writes the verdicts to `results/make_it_fail.txt`.

In [5]:
lines = [verdict_line(label, s, r) for label, s, r in verdicts]
print('\n'.join(lines))
expected = {"1a": True, "1b": False, "2:": False, "2'": True, "3: owlrl, hand-listed V": False, "3: owlrl, V_full": True}
for label, s, r in verdicts:
    key = next(k for k in expected if label.startswith(k))
    assert (s == r) == expected[key], label
show.write_log('../results/make_it_fail.txt', lines +
               [f"summary: {len(verdicts)} rows, {sum(s != r for _, s, r in verdicts)} disagreements, all as predicted"])

1a: Cor. 3 example, all instances        semantics True   rules True   agree
1b: Cor. 3 example, instances over N only semantics True   rules False  DISAGREE
2: non-uniform regime                    semantics False  rules True   DISAGREE
2': same rule, uniform                   semantics True   rules True   agree
3: owlrl, hand-listed V                  semantics False  rules True   DISAGREE
3: owlrl, V_full                         semantics True   rules True   agree
wrote ../results/make_it_fail.txt (7 lines)
